# How PSO searches cosmological parameter space

This notebook isolates the optimization mechanics behind the cosmology fits. It follows the method used in the project presentation: define a cosmological objective, identify personal and global best positions, update the swarm, and track convergence toward the minimum.

For flat $\Lambda$CDM, each particle represents a candidate universe

$$
\mathbf{x}_j=(H_{0,j},\Omega_{m,j}),
$$

and is scored with the 31-point cosmic-chronometer objective

$$
\chi^2(\boldsymbol\theta)=\sum_i\frac{[H_{\rm model}(z_i;\boldsymbol\theta)-H_i]^2}{\sigma_i^2}.
$$

The PSO update is

$$
\mathbf{v}_j^{(t+1)}=\omega\mathbf{v}_j^{(t)}
+c_1\mathbf{r}_1\odot(\mathbf{p}_j-\mathbf{x}_j^{(t)})
+c_2\mathbf{r}_2\odot(\mathbf{g}-\mathbf{x}_j^{(t)}),
$$

$$
\mathbf{x}_j^{(t+1)}=\mathbf{x}_j^{(t)}+\mathbf{v}_j^{(t+1)}.
$$

Here $\mathbf{p}_j$ is the particle's personal best (`pbest`) and $\mathbf{g}$ is the swarm's global best (`gbest`).

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from pso_cosmology.data import load_cosmic_chronometers
from pso_cosmology.models import h_flat_lcdm
from pso_cosmology.pso import run_pso
from pso_cosmology.statistics import chi_square

QUICK = os.environ.get("PSO_QUICK", "0") == "1"
FIGURE_DIR = Path(os.environ.get("PSO_OUTPUT_DIR", ROOT / "figures" / "method"))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

z, H_obs, sigma_H = load_cosmic_chronometers(ROOT / "data" / "cosmic_chronometers_31.txt")

def objective(theta):
    H0, omega_m = theta
    if not (50.0 <= H0 <= 100.0 and 0.1 <= omega_m <= 0.9):
        return np.inf
    return chi_square(h_flat_lcdm(z, H0, omega_m), H_obs, sigma_H)

def chi2_grid(n=150):
    H0_grid = np.linspace(50.0, 100.0, n)
    om_grid = np.linspace(0.1, 0.9, n)
    HH, OO = np.meshgrid(H0_grid, om_grid)
    model = HH[..., None] * np.sqrt(OO[..., None] * (1.0 + z)**3 + (1.0 - OO[..., None]))
    chi = np.sum(((model - H_obs) / sigma_H) ** 2, axis=-1)
    return H0_grid, om_grid, HH, OO, chi


## Five candidate universes

The small example below reproduces the numerical illustration used in the presentation. It uses the same $H(z)$ data and the same $\chi^2$ objective as the full calculation.

In [ ]:
labels = np.array(["P1", "P2", "P3", "P4", "P5"])
initial_demo = np.array([
    [60.0, 0.20],
    [68.0, 0.30],
    [75.0, 0.50],
    [85.0, 0.20],
    [55.0, 0.70],
])
updated_demo = np.array([
    [70.976, 0.3187],
    [68.144, 0.3144],
    [68.022, 0.2328],
    [60.763, 0.2966],
    [69.279, 0.3876],
])
initial_scores = np.array([objective(p) for p in initial_demo])
updated_scores = np.array([objective(p) for p in updated_demo])
initial_gbest = int(np.argmin(initial_scores))
updated_gbest = int(np.argmin(updated_scores))

print("Initial scores:", np.round(initial_scores, 2))
print("After one illustrative update:", np.round(updated_scores, 2))
print("Initial gbest:", labels[initial_gbest], initial_demo[initial_gbest])
print("Updated gbest:", labels[updated_gbest], updated_demo[updated_gbest])


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(initial_demo[:, 0], initial_demo[:, 1], s=75, label="Initial candidate universes")
for label, (H0, om), score in zip(labels, initial_demo, initial_scores):
    ax.annotate(f"{label}\n$\\chi^2$={score:.2f}", (H0, om), xytext=(6, 6), textcoords="offset points")
ax.scatter(initial_demo[initial_gbest, 0], initial_demo[initial_gbest, 1], s=180, marker="*", label="Initial global best")
ax.set(xlim=(50, 90), ylim=(0.1, 0.75), xlabel=r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]", ylabel=r"$\Omega_m$", title="Five candidate cosmologies and the first global best")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "five_particle_initial_gbest.png", dpi=220, bbox_inches="tight")
plt.show()


## One illustrative iteration

At initialization each particle's `pbest` is its starting position. The arrows show the movement used in the presentation example. After moving, the same $\chi^2$ objective is evaluated again and the best positions are updated.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(initial_demo[:, 0], initial_demo[:, 1], s=70, alpha=0.75, label="Before update")
ax.scatter(updated_demo[:, 0], updated_demo[:, 1], s=80, marker="^", label="After update")
for i, label in enumerate(labels):
    ax.annotate("", xy=updated_demo[i], xytext=initial_demo[i], arrowprops=dict(arrowstyle="->", lw=1.5, alpha=0.75))
    ax.annotate(label, updated_demo[i], xytext=(5, 5), textcoords="offset points")
ax.scatter(updated_demo[updated_gbest, 0], updated_demo[updated_gbest, 1], s=200, marker="*", label="New global best")
ax.set(xlim=(50, 90), ylim=(0.1, 0.75), xlabel=r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]", ylabel=r"$\Omega_m$", title="One PSO iteration: candidate universes move and are rescored")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "five_particle_one_iteration.png", dpi=220, bbox_inches="tight")
plt.show()


## Production swarm

The remaining figures use the actual production configuration: 100 particles, 300 iterations, $\omega=0.72$, $c_1=c_2=1.49$, and seed 42. Quick mode reduces the swarm for execution testing but uses the same diagnostics.

In [ ]:
settings = dict(
    n_particles=40 if QUICK else 100,
    n_iter=60 if QUICK else 300,
    inertia=0.72,
    cognitive=1.49,
    social=1.49,
    seed=42,
)
result = run_pso(objective, [(50.0, 100.0), (0.1, 0.9)], **settings)
print("Best position:", result.best_position)
print("Best chi-square:", result.best_score)


## Personal memory and global memory

The next diagnostic shows both attractions at an intermediate iteration. The `pbest` arrow points toward a particle's own best-known location. The `gbest` arrow points toward the best location discovered by the full swarm. These are the cognitive and social search components in the update equation.

In [ ]:
step = min(5, result.positions_history.shape[0] - 1)
current = result.positions_history[step]
pbest = result.personal_best_history[step]
gbest = result.global_best_position_history[max(step - 1, 0)]

# Choose a deterministic subset distributed through particle index space.
count = min(10, len(current))
indices = np.linspace(0, len(current) - 1, count, dtype=int)

H0_grid, om_grid, HH, OO, chi = chi2_grid(110 if QUICK else 170)
levels = np.quantile(chi[np.isfinite(chi)], [0.02, 0.05, 0.10, 0.20, 0.40])
levels = np.unique(levels)

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.contour(HH, OO, chi, levels=levels, linewidths=0.9, alpha=0.45)
ax.scatter(current[indices, 0], current[indices, 1], s=45, label="Current particle position", zorder=4)
ax.scatter(pbest[indices, 0], pbest[indices, 1], s=35, marker="s", label="Personal best", zorder=4)
ax.scatter(gbest[0], gbest[1], s=190, marker="*", label="Global best", zorder=5)

for idx in indices:
    ax.annotate("", xy=pbest[idx], xytext=current[idx], arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.75))
    ax.annotate("", xy=gbest, xytext=current[idx], arrowprops=dict(arrowstyle="->", lw=1.0, ls="--", alpha=0.50))

handles, labels_ = ax.get_legend_handles_labels()
handles.extend([
    Line2D([0], [0], lw=1.4, label="Cognitive pull: current → pbest"),
    Line2D([0], [0], lw=1.2, ls="--", label="Social pull: current → gbest"),
])
labels_.extend(["Cognitive pull: current → pbest", "Social pull: current → gbest"])
ax.legend(handles, labels_, fontsize=9, loc="upper right")
ax.set(xlim=(50, 100), ylim=(0.1, 0.9), xlabel=r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]", ylabel=r"$\Omega_m$", title=f"How PSO hunts: personal and global memory after {step} iterations")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pbest_gbest_search.png", dpi=220, bbox_inches="tight")
plt.show()


## Iteration snapshots

A direct validation of the optimization is to inspect the swarm at several iterations on the same $\chi^2(H_0,\Omega_m)$ surface. The particles should migrate from the broad initialization box into the low-$\chi^2$ valley.

In [ ]:
snapshot_steps = [0, 1, min(10, settings["n_iter"]), min(50, settings["n_iter"])]
snapshot_steps = list(dict.fromkeys(snapshot_steps))

H0_grid, om_grid, HH, OO, chi = chi2_grid(100 if QUICK else 180)
chi_min = np.nanmin(chi)
contour_levels = chi_min + np.array([0.5, 2.0, 5.0, 15.0, 40.0, 100.0])

fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(4.2 * len(snapshot_steps), 4.1), sharex=True, sharey=True)
if len(snapshot_steps) == 1:
    axes = [axes]

for ax, step_i in zip(axes, snapshot_steps):
    pts = result.positions_history[step_i]
    ax.contour(HH, OO, chi, levels=contour_levels, linewidths=0.8, alpha=0.55)
    ax.scatter(pts[:, 0], pts[:, 1], s=15, alpha=0.55)
    if step_i == 0:
        scores = np.array([objective(p) for p in pts])
        best_idx = int(np.argmin(scores))
        best_pos = pts[best_idx]
        best_score = scores[best_idx]
    else:
        best_pos = result.global_best_position_history[step_i - 1]
        best_score = result.best_score_history[step_i - 1]
    ax.scatter(best_pos[0], best_pos[1], s=110, marker="*", zorder=5)
    ax.set_title(f"Iteration {step_i}\nbest $\\chi^2$={best_score:.3f}")
    ax.set_xlabel(r"$H_0$")
axes[0].set_ylabel(r"$\Omega_m$")
fig.suptitle("PSO swarm contracts into the low-$\\chi^2$ valley", y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "iteration_snapshots.png", dpi=220, bbox_inches="tight")
plt.show()


## Global-best path and convergence

The global-best path records the best-known cosmology after each update. Its movement becomes progressively smaller as the swarm settles near the minimum. The best-$\chi^2$ history is non-increasing because a new global best is accepted only when it improves the objective.

In [ ]:
gb = result.global_best_position_history
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.contour(HH, OO, chi, levels=contour_levels, linewidths=0.9, alpha=0.55)
ax.scatter(result.initial_positions[:, 0], result.initial_positions[:, 1], s=18, alpha=0.20, label="Initial swarm")
ax.plot(gb[:, 0], gb[:, 1], "-o", ms=2.5, lw=1.8, label="Global-best path")
ax.scatter(gb[0, 0], gb[0, 1], s=90, marker="s", label="First global best")
ax.scatter(gb[-1, 0], gb[-1, 1], s=150, marker="*", label="Final global best")
ax.set(xlim=(50, 100), ylim=(0.1, 0.9), xlabel=r"$H_0$ [km s$^{-1}$ Mpc$^{-1}$]", ylabel=r"$\Omega_m$", title="Evolution of the global-best cosmology")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "global_best_path.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.plot(np.arange(1, len(result.best_score_history) + 1), result.best_score_history, lw=2.2)
ax.axhline(result.best_score, ls="--", lw=1.2, label=f"Final $\\chi^2$ = {result.best_score:.6f}")
ax.set(xlabel="PSO iteration", ylabel=r"Best $\chi^2$ found so far", title="Convergence of the global-best objective")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "global_best_convergence.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
iterations = np.arange(1, len(gb) + 1)
axes[0].plot(iterations, gb[:, 0], lw=2)
axes[0].axhline(result.best_position[0], ls="--", lw=1)
axes[0].set_ylabel(r"Best $H_0$")
axes[0].set_title("Best-fit parameters stabilize with iteration")
axes[1].plot(iterations, gb[:, 1], lw=2)
axes[1].axhline(result.best_position[1], ls="--", lw=1)
axes[1].set(xlabel="PSO iteration", ylabel=r"Best $\Omega_m$")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "global_best_parameter_history.png", dpi=220, bbox_inches="tight")
plt.show()
